# Phase 6 — Decision Tree and Random Forest
Goal: test whether tree models beat the linear SVM (honest cost 45,100).
Preprocessing: no log, no scaling (trees don't need them).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Imports from src
from src.data_loader import load_aps_data
from src.preprocessing import (
    build_linear_preprocessor,
    build_tree_preprocessor,
    make_cv_splitter,
)
from src.evaluation import (
    calculate_aps_cost,
    calculate_classification_metrics,
    generate_oof_predictions,
    best_cost_threshold,
    oof_scores,
    honest_cost,
    robust_cv_predictions,
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


In [2]:
# Define paths and load the dataset
RES = PROJECT_ROOT / "reports" / "results"
RES.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_ROOT / "dataset"
X, y = load_aps_data(DATA_DIR / "aps_failure_training_set.csv")
cv = make_cv_splitter()

print(f"Loaded dataset: X shape = {X.shape}, y distribution = {dict(y.value_counts())}")


Loaded dataset: X shape = (60000, 170), y distribution = {0: np.int64(59000), 1: np.int64(1000)}


### Hypotheses & Predictions (before running):
1. The best single decision tree will cost more than the linear SVM baseline (45,100) due to high variance and axis-aligned step boundaries.
2. Random Forest ensembles will significantly reduce variance, improve probability calibration, and lower the honest misclassification cost.

In [3]:
import numpy as np
import pandas as pd

def make_tree(max_depth, min_leaf, class_weight=None):
    return Pipeline([
        ("prep", build_tree_preprocessor()),
        ("tree", DecisionTreeClassifier(
            max_depth=max_depth,
            min_samples_leaf=min_leaf,
            class_weight=class_weight,
            random_state=42
        )),
    ])

tree_rows, tree_scores = [], {}
for depth in [3, 5, 8, 12, None]:
    for leaf in [5, 50]:
        s, _, secs = oof_scores(make_tree(depth, leaf), X, y, cv)
        tree_scores[(depth, leaf)] = s
        tuned = best_cost_threshold(y, s)
        h_cost, h_fp, h_fn = honest_cost(y, s, cv)
        tree_rows.append({
            "model": "DecisionTree",
            "max_depth": depth,
            "min_samples_leaf": leaf,
            "seconds": round(secs, 1),
            "distinct_scores": len(np.unique(s)),
            "roc_auc": roc_auc_score(y, s),
            "pr_auc": average_precision_score(y, s),
            "tuned_cost": tuned["aps_cost"],
            "honest_cost": h_cost,
            "honest_fp": h_fp,
            "honest_fn": h_fn,
        })
        print(f"depth={depth} leaf={leaf} -> honest {h_cost} ({secs:.1f}s)")

tree_results = pd.DataFrame(tree_rows).sort_values("honest_cost").reset_index(drop=True)
tree_results.to_csv(RES / "phase6_tree_results.csv", index=False)
tree_results.round(4)


depth=3 leaf=5 -> honest 91540 (41.8s)
depth=3 leaf=50 -> honest 91540 (55.9s)
depth=5 leaf=5 -> honest 90280 (55.1s)
depth=5 leaf=50 -> honest 77630 (71.4s)
depth=8 leaf=5 -> honest 86800 (7657.0s)
depth=8 leaf=50 -> honest 59520 (51.7s)
depth=12 leaf=5 -> honest 97120 (66.9s)
depth=12 leaf=50 -> honest 64870 (70.8s)
depth=None leaf=5 -> honest 115480 (117.7s)
depth=None leaf=50 -> honest 65630 (97.4s)


,model,max_depth,min_samples_leaf,seconds,distinct_scores,roc_auc,pr_auc,tuned_cost,honest_cost,honest_fp,honest_fn
0,DecisionTree,8.0,50,51.7,134,0.9627,0.7177,58920,59520,2652,66
1,DecisionTree,12.0,50,70.8,154,0.9463,0.7182,63550,64870,2437,81
2,DecisionTree,NaN,50,97.4,161,0.9587,0.7209,63540,65630,2763,76
3,DecisionTree,5.0,50,71.4,81,0.9396,0.6816,77090,77630,1763,120
4,DecisionTree,8.0,5,7657.0,79,0.9053,0.6952,86710,86800,1630,141
5,DecisionTree,5.0,5,55.1,68,0.9188,0.6772,85000,90280,1228,156
6,DecisionTree,3.0,5,41.8,38,0.9207,0.6259,91120,91540,1254,158
7,DecisionTree,3.0,50,55.9,40,0.9206,0.6130,91120,91540,1254,158
8,DecisionTree,12.0,5,66.9,61,0.8808,0.6630,96200,97120,1962,155
9,DecisionTree,NaN,5,117.7,28,0.8892,0.6385,115030,115480,698,217


## Random Forest Benchmark
Ensemble trees with feature subsampling and bootstrap aggregating.

In [4]:
def make_rf(n_estimators=100, max_depth=None, min_leaf=5, class_weight="balanced"):
    return Pipeline([
        ("prep", build_tree_preprocessor()),
        ("rf", RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_leaf,
            class_weight=class_weight,
            random_state=42,
            n_jobs=-1
        )),
    ])

# Random Forest Benchmark
rf_scores, _, rf_secs = oof_scores(make_rf(n_estimators=100, max_depth=12, min_leaf=5), X, y, cv)
rf_tuned = best_cost_threshold(y, rf_scores)
rf_cost, rf_fp, rf_fn = honest_cost(y, rf_scores, cv)

print(f"Random Forest (100 trees, depth 12): honest cost = {rf_cost} (FP={rf_fp}, FN={rf_fn}, {rf_secs:.1f}s)")


Random Forest (100 trees, depth 12): honest cost = 43230 (FP=2273, FN=41, 74.5s)
